# Steam 인디게임 기술통계 노트북
- `steam_stratified_sample_v4.csv`
- `review_histogram_v4.csv`
- `steam_indie_list_202604211615.csv`
- `steam_indie_reviews_202604230927.csv`

In [32]:
from pathlib import Path
import ast
import pandas as pd
import numpy as np
from IPython.display import display

## 1. 파일 경로 설정

In [33]:
# 프로젝트 루트 직접 지정
ROOT = Path(r"C:\Users\joon5\Documents\github\steam-indie-game-analysis")

# data/processed 폴더
DATA_DIR = ROOT / "data" / "processed"

# 파일 경로
HIST_PATH   = DATA_DIR / "review_histogram_v4.csv"
FULL_PATH   = DATA_DIR / "steam_indie_list_202604211615.csv"
REVIEWS_PATH = DATA_DIR / "steam_indie_reviews_202604230927.csv"


# 확인
print("ROOT        =", ROOT)
print("DATA_DIR    =", DATA_DIR)
print("HIST_PATH   =", HIST_PATH)
print("FULL_PATH   =", FULL_PATH)

print("hist exists   :", HIST_PATH.exists())
print("full exists   :", FULL_PATH.exists())

# 읽기
hist_df = pd.read_csv(HIST_PATH)
full_df = pd.read_csv(FULL_PATH)
reviews_df = pd.read_csv(REVIEWS_PATH)

print("hist_df shape   :", hist_df.shape)
print("full_df shape   :", full_df.shape)
print("reviews_df shape:", reviews_df.shape)

ROOT        = C:\Users\joon5\Documents\github\steam-indie-game-analysis
DATA_DIR    = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed
HIST_PATH   = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\review_histogram_v4.csv
FULL_PATH   = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_indie_list_202604211615.csv
hist exists   : True
full exists   : True
hist_df shape   : (6003, 9)
full_df shape   : (61266, 12)
reviews_df shape: (13106, 21)


In [34]:
print("[hist_df columns]")
print(hist_df.columns.tolist())
print()

print("[full_df columns]")
print(full_df.columns.tolist())
print()

print("[reviews_df columns]")
print(reviews_df.columns.tolist())

[hist_df columns]
['appid', 'name', 'stratum', 'release_date', 'date', 'recommendations_up', 'recommendations_down', 'rollup_type', 'data_type']

[full_df columns]
['appid', 'spy_name', 'owners', 'positive', 'negative', 'price_spy', 'ccu', 'name_store', 'type', 'genres', 'release_date', 'developers']

[reviews_df columns]
['recommendationid', 'appid', 'language', 'review', 'timestamp_created', 'timestamp_updated', 'voted_up', 'votes_up', 'votes_funny', 'weighted_vote_score', 'comment_count', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'author_steamid', 'author_num_games_owned', 'author_num_reviews', 'author_playtime_forever', 'author_playtime_last_two_weeks', 'author_playtime_at_review', 'author_last_played']


# CSV별 전처리

## `hist_df` 전처리

In [35]:
hist_df = hist_df.copy()

# release_date 변환
hist_df['release_date'] = pd.to_datetime(hist_df['release_date'], errors='coerce')

# date는 epoch second -> datetime
hist_df['date_dt'] = pd.to_datetime(hist_df['date'], unit='s', errors='coerce')

# 총 추천 수
hist_df['total_recommendations'] = (
    hist_df['recommendations_up'].fillna(0) + hist_df['recommendations_down'].fillna(0)
)

# 긍정률
hist_df['positive_ratio'] = np.where(
    hist_df['total_recommendations'] > 0,
    hist_df['recommendations_up'] / hist_df['total_recommendations'],
    np.nan
)
hist_df['positive_ratio_pct'] = (hist_df['positive_ratio'] * 100).round(2)

# 연/월 파생
hist_df['release_year'] = hist_df['release_date'].dt.year
hist_df['review_year'] = hist_df['date_dt'].dt.year
hist_df['review_month'] = hist_df['date_dt'].dt.month
hist_df['year_month'] = hist_df['date_dt'].dt.to_period('M').astype(str)

print(hist_df.shape)
display(hist_df.head())


(6003, 17)


,appid,name,stratum,release_date,date,recommendations_up,recommendations_down,rollup_type,data_type,date_dt,total_recommendations,positive_ratio,positive_ratio_pct,release_year,review_year,review_month,year_month
0,1432860,Sun Haven,large_high,2023-03-10,1622505600,280,34,month,rollups,2021-06-01,314,0.891720,89.17,2023,2021,6,2021-06
1,1432860,Sun Haven,large_high,2023-03-10,1625097600,448,46,month,rollups,2021-07-01,494,0.906883,90.69,2023,2021,7,2021-07
2,1432860,Sun Haven,large_high,2023-03-10,1627776000,170,27,month,rollups,2021-08-01,197,0.862944,86.29,2023,2021,8,2021-08
3,1432860,Sun Haven,large_high,2023-03-10,1630454400,112,18,month,rollups,2021-09-01,130,0.861538,86.15,2023,2021,9,2021-09
4,1432860,Sun Haven,large_high,2023-03-10,1633046400,116,10,month,rollups,2021-10-01,126,0.920635,92.06,2023,2021,10,2021-10


## `full_df` 전처리

In [36]:
full_df = full_df.copy()

# release_date 변환
full_df['release_date_dt'] = pd.to_datetime(full_df['release_date'], errors='coerce')

# genres 문자열 -> 리스트 변환
full_df['genres_list'] = full_df['genres'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

# 장르 개수
full_df['genre_count'] = full_df['genres_list'].apply(len)

# 총 리뷰 수
full_df['total_reviews'] = full_df['positive'].fillna(0) + full_df['negative'].fillna(0)

# 긍정률
full_df['positive_ratio'] = np.where(
    full_df['total_reviews'] > 0,
    full_df['positive'] / full_df['total_reviews'],
    np.nan
)
full_df['positive_ratio_pct'] = (full_df['positive_ratio'] * 100).round(2)

# 출시 연도
full_df['release_year'] = full_df['release_date_dt'].dt.year

# owners 범위 문자열에서 하한/상한 추출
owners_split = full_df['owners'].astype(str).str.replace(',', '', regex=False).str.split(' .. ', expand=True)
full_df['owners_lower_calc'] = pd.to_numeric(owners_split[0], errors='coerce')
full_df['owners_upper_calc'] = pd.to_numeric(owners_split[1], errors='coerce')

# owners 중앙값 느낌의 보조 컬럼
full_df['owners_mid_calc'] = (full_df['owners_lower_calc'] + full_df['owners_upper_calc']) / 2

# 무료 여부 보조컬럼
full_df['is_free_by_price'] = full_df['price_spy'].fillna(-1).eq(0)

print(full_df.shape)
display(full_df.head())


(61266, 23)


,appid,spy_name,owners,positive,negative,price_spy,ccu,name_store,type,genres,...,genres_list,genre_count,total_reviews,positive_ratio,positive_ratio_pct,release_year,owners_lower_calc,owners_upper_calc,owners_mid_calc,is_free_by_price
0,1623730,Palworld,"50,000,000 .. 100,000,000",358266,22443,2999,18028,Palworld,game,"['Action', 'Adventure', 'Indie', 'RPG', 'Early...",...,"[Action, Adventure, Indie, RPG, Early Access]",5,380709,0.941049,94.10,2024.0,50000000,100000000,75000000.0,False
1,304930,Unturned,"50,000,000 .. 100,000,000",506516,48852,0,10408,Unturned,game,"['Action', 'Adventure', 'Casual', 'Indie', 'Fr...",...,"[Action, Adventure, Casual, Indie, Free To Play]",5,555368,0.912037,91.20,2017.0,50000000,100000000,75000000.0,True
2,105600,Terraria,"20,000,000 .. 50,000,000",1373979,35494,999,24580,Terraria,game,"['Action', 'Adventure', 'Indie', 'RPG']",...,"[Action, Adventure, Indie, RPG]",4,1409473,0.974818,97.48,2011.0,20000000,50000000,35000000.0,False
3,431960,Wallpaper Engine,"20,000,000 .. 50,000,000",876898,17560,499,91184,Wallpaper Engine,game,"['Casual', 'Indie', 'Animation & Modeling', 'D...",...,"[Casual, Indie, Animation & Modeling, Design &...",6,894458,0.980368,98.04,2018.0,20000000,50000000,35000000.0,False
4,291550,Brawlhalla,"20,000,000 .. 50,000,000",314809,71647,0,14169,Brawlhalla,game,"['Action', 'Indie', 'Free To Play']",...,"[Action, Indie, Free To Play]",3,386456,0.814605,81.46,2017.0,20000000,50000000,35000000.0,True


## `reviews_df` 전처리

In [37]:
reviews_df = reviews_df.copy()

# timestamp -> datetime
reviews_df['timestamp_created_dt'] = pd.to_datetime(reviews_df['timestamp_created'], unit='s', errors='coerce')
reviews_df['timestamp_updated_dt'] = pd.to_datetime(reviews_df['timestamp_updated'], unit='s', errors='coerce')
reviews_df['author_last_played_dt'] = pd.to_datetime(reviews_df['author_last_played'], unit='s', errors='coerce')

# 리뷰 길이
reviews_df['review_char_len'] = reviews_df['review'].fillna('').str.len()
reviews_df['review_word_count'] = reviews_df['review'].fillna('').str.split().str.len()

# 플레이타임(시간 단위)
reviews_df['author_playtime_forever_hour'] = (reviews_df['author_playtime_forever'] / 60).round(2)
reviews_df['author_playtime_last_two_weeks_hour'] = (reviews_df['author_playtime_last_two_weeks'] / 60).round(2)
reviews_df['author_playtime_at_review_hour'] = (reviews_df['author_playtime_at_review'] / 60).round(2)

# 추천 여부 숫자형
reviews_df['voted_up_int'] = reviews_df['voted_up'].astype('Int64')

# 리뷰 작성 연/월
reviews_df['review_year'] = reviews_df['timestamp_created_dt'].dt.year
reviews_df['review_month'] = reviews_df['timestamp_created_dt'].dt.month
reviews_df['review_year_month'] = reviews_df['timestamp_created_dt'].dt.to_period('M').astype(str)

print(reviews_df.shape)
display(reviews_df.head())

(13106, 33)


,recommendationid,appid,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,...,author_last_played_dt,review_char_len,review_word_count,author_playtime_forever_hour,author_playtime_last_two_weeks_hour,author_playtime_at_review_hour,voted_up_int,review_year,review_month,review_year_month
0,134554560,1432860,english,"For me, Sun Haven is a fun farming game that h...",1678643330,1678643330,True,1432,25,0.971922,...,2025-11-24 06:56:10,652,133,68.67,0.0,28.72,1,2023,3,2023-03
1,135610350,1432860,english,*Updated for v1.0.3*\n\nI really love this gam...,1680018629,1680192211,False,1564,18,0.939696,...,2023-07-22 18:23:39,7710,1347,215.85,0.0,143.42,0,2023,3,2023-03
2,136483272,1432860,english,(Singleplayer review)\nPuh i am not really sur...,1681149417,1681149417,True,166,0,0.917131,...,2024-08-15 13:36:02,1317,247,60.88,0.0,56.67,1,2023,4,2023-04
3,138973732,1432860,russian,"Я наиграла в эту игру почти 55 часов, 40+ из к...",1685034459,1685034459,False,224,7,0.897391,...,2023-05-19 22:07:03,7999,1271,54.67,0.0,54.67,0,2023,5,2023-05
4,136801481,1432860,japanese,Sun Haven好きさんが増えると嬉しいので、初めてレビューします。\n\nこれを見たあな...,1681649431,1682154116,True,143,10,0.895801,...,2024-08-12 15:19:36,1257,53,200.95,0.0,130.82,1,2023,4,2023-04


# 기술통계

In [38]:
# 컬럼 정보 간단 확인
def check_basic_info(df, df_name, exclude_cols=None):
    print(f"\n{'='*80}")
    print(f"{df_name}의 컬럼 정보 / 결측치 확인 정보 요약")
    print(f"{'='*80}\n")

    df_copied = df.copy()

    if exclude_cols:
        df_copied = df_copied.drop(columns=exclude_cols, errors='ignore')

    for col in df_copied.columns:
        try:
            df_copied[col].nunique(dropna=True)
        except TypeError:
            df_copied[col] = df_copied[col].astype(str)

    overview_df = pd.DataFrame({
        '항목': ['행 개수', '열 개수', '중복 행 개수'],
        '값': [df_copied.shape[0], df_copied.shape[1], df_copied.duplicated().sum()]
    })

    summary_df = pd.DataFrame({
        '데이터타입': df_copied.dtypes.astype(str),
        '행 개수': df_copied.count(),
        '행 비율(%)': (df_copied.count() / len(df_copied) * 100).round(2),
        '결측치 개수': df_copied.isnull().sum(),
        '결측치 비율(%)': (df_copied.isnull().sum() / len(df_copied) * 100).round(2),
        '고유값 개수': df_copied.nunique(dropna=True)
    }).sort_values(by=['결측치 개수', '고유값 개수'], ascending=[False, False])

    print("[전체 요약]")
    display(overview_df)

    print("[컬럼별 요약]")
    display(summary_df)

    print("[상위 5행]")
    display(df_copied.head())

In [39]:
# ID 컬럼 중복 확인
def check_id_duplicates(df, col_name, df_name):
    print(f"\n{'='*80}")
    print(f"{df_name}의 값 중복 확인")
    print(f"{'='*80}")

    df_copied = df.copy()

    if col_name not in df_copied.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    print("전체 행 수:", len(df_copied))
    print(f"{col_name} 고유 개수:", df_copied[col_name].nunique(dropna=True))
    print(f"중복 {col_name} 개수:", df_copied[col_name].duplicated().sum())

    display(df_copied[[col_name]].head())

In [40]:
# 범주형 컬럼 분포 확인
def check_category_summary(df, df_name, col_name, top_n=10):
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 범주 확인")
    print(f"{'='*80}")

    df_copied = df.copy()

    if col_name not in df_copied.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    summary_df = df_copied[col_name].value_counts(dropna=False).reset_index()
    summary_df.columns = [col_name, '개수']
    summary_df['비율(%)'] = (summary_df['개수'] / len(df_copied) * 100).round(2)

    print("전체 행 수:", len(df_copied))
    print(f"{col_name} 고유값 개수(결측 포함):", df_copied[col_name].nunique(dropna=False))
    print()

    display(summary_df.head(top_n))

In [41]:
# 불리언 컬럼 분포 확인
def check_boolean_summary(df, df_name, col_name):
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 불리언 분포 확인")
    print(f"{'='*80}")

    if col_name not in df.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    summary_df = df[col_name].value_counts(dropna=False).reset_index()
    summary_df.columns = [col_name, '개수']
    summary_df['비율(%)'] = (summary_df['개수'] / len(df) * 100).round(2)

    display(summary_df)

In [42]:
# 수치형 컬럼 기술통계
def check_numeric_summary(df, df_name, cols=None):
    print(f"\n{'='*80}")
    print(f"{df_name}의 수치형 기술통계")
    print(f"{'='*80}")

    if cols is None:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    else:
        numeric_cols = [col for col in cols if col in df.columns]

    if len(numeric_cols) == 0:
        print("수치형 컬럼이 없습니다.")
        return

    summary_df = df[numeric_cols].describe().T
    summary_df['결측치 개수'] = df[numeric_cols].isnull().sum()
    summary_df['왜도'] = df[numeric_cols].skew(numeric_only=True)
    summary_df['첨도'] = df[numeric_cols].kurt(numeric_only=True)

    display(summary_df)

In [43]:
# 날짜형 컬럼 범위 확인
def check_datetime_summary(df, df_name, cols=None):
    print(f"\n{'='*80}")
    print(f"{df_name}의 날짜형 컬럼 요약")
    print(f"{'='*80}")

    if cols is None:
        dt_cols = df.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns.tolist()
    else:
        dt_cols = [col for col in cols if col in df.columns]

    if len(dt_cols) == 0:
        print("날짜형 컬럼이 없습니다.")
        return

    rows = []
    for col in dt_cols:
        rows.append({
            '컬럼명': col,
            '결측치 개수': df[col].isnull().sum(),
            '최소값': df[col].min(),
            '최대값': df[col].max(),
            '고유값 개수': df[col].nunique(dropna=True)
        })

    summary_df = pd.DataFrame(rows)
    display(summary_df)

In [44]:
# 리스트 컬럼 펼쳐서 분포 확인
def check_list_column_summary(df, df_name, col_name, top_n=20):
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 리스트 항목 분포")
    print(f"{'='*80}")

    if col_name not in df.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    temp = df[col_name].dropna()

    exploded = temp.explode()
    summary_df = exploded.value_counts(dropna=False).reset_index()
    summary_df.columns = [col_name, '개수']
    summary_df['비율(%)'] = (summary_df['개수'] / len(df) * 100).round(2)

    display(summary_df.head(top_n))

In [45]:
# 텍스트 컬럼 길이 확인 (필요한가?)
def check_text_summary(df, df_name, col_name):
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 텍스트 길이 요약")
    print(f"{'='*80}")

    if col_name not in df.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    temp = df[col_name].fillna('').astype(str)

    summary_df = pd.DataFrame({
        '항목': ['결측치 개수', '평균 글자수', '중앙값 글자수', '최대 글자수', '최소 글자수'],
        '값': [
            df[col_name].isnull().sum(),
            round(temp.str.len().mean(), 2),
            temp.str.len().median(),
            temp.str.len().max(),
            temp.str.len().min()
        ]
    })

    display(summary_df)

# 데이터셋별 기술통계 실행

## `hist_df` 기술통계

In [46]:
check_basic_info(hist_df, 'hist_df')
check_id_duplicates(hist_df, 'appid', 'hist_df')
check_category_summary(hist_df, 'hist_df', 'stratum')
check_category_summary(hist_df, 'hist_df', 'rollup_type')
check_category_summary(hist_df, 'hist_df', 'data_type')
check_numeric_summary(
    hist_df,
    'hist_df',
    cols=['recommendations_up', 'recommendations_down', 'total_recommendations', 'positive_ratio_pct']
)
check_datetime_summary(hist_df, 'hist_df', cols=['release_date', 'date_dt'])


hist_df의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,6003
1,열 개수,17
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
positive_ratio,float64,4708,78.43,1295,21.57,1159
positive_ratio_pct,float64,4708,78.43,1295,21.57,1032
date,int64,6003,100.00,0,0.00,597
date_dt,datetime64[s],6003,100.00,0,0.00,597
total_recommendations,int64,6003,100.00,0,0.00,586
recommendations_up,int64,6003,100.00,0,0.00,560
recommendations_down,int64,6003,100.00,0,0.00,199
year_month,str,6003,100.00,0,0.00,90
appid,int64,6003,100.00,0,0.00,74
name,str,6003,100.00,0,0.00,74


[상위 5행]


...

,data_type,개수,비율(%)
0,rollups,3783,63.02
1,recent,2220,36.98



hist_df의 수치형 기술통계


,count,mean,std,min,25%,50%,75%,max,결측치 개수,왜도,첨도
recommendations_up,6003.0,111.393637,820.364663,0.0,0.0,5.000,28.0,29749.0,0,20.335203,536.396106
recommendations_down,6003.0,10.292021,45.791930,0.0,0.0,1.000,4.0,1231.0,0,11.964448,213.640398
total_recommendations,6003.0,121.685657,841.672838,0.0,1.0,6.000,33.0,30132.0,0,19.739715,511.010027
positive_ratio_pct,4708.0,81.836177,24.356116,0.0,75.0,90.765,100.0,100.0,1295,-1.922447,3.495948



hist_df의 날짜형 컬럼 요약


,컬럼명,결측치 개수,최소값,최대값,고유값 개수
0,release_date,0,2023-02-02,2025-10-16,72
1,date_dt,0,2018-11-01,2026-04-20,597


## `full_df` 기술통계

In [47]:
check_basic_info(full_df, 'full_df')
check_id_duplicates(full_df, 'appid', 'full_df')
check_category_summary(full_df, 'full_df', 'type')
check_category_summary(full_df, 'full_df', 'is_free_by_price')
check_numeric_summary(
    full_df,
    'full_df',
    cols=['positive', 'negative', 'total_reviews', 'price_spy', 'ccu',
          'genre_count', 'positive_ratio_pct', 'owners_lower_calc', 'owners_upper_calc', 'owners_mid_calc']
)
check_datetime_summary(full_df, 'full_df', cols=['release_date_dt'])
check_list_column_summary(full_df, 'full_df', 'genres_list')



full_df의 컬럼 정보 / 결측치 확인 정보 요약



[전체 요약]


,항목,값
0,행 개수,61266
1,열 개수,23
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
positive_ratio,float64,60897,99.40,369,0.60,12982
positive_ratio_pct,float64,60897,99.40,369,0.60,5317
release_date_dt,datetime64[us],61109,99.74,157,0.26,4598
release_year,float64,61109,99.74,157,0.26,26
developers,str,61170,99.84,96,0.16,41840
release_date,str,61233,99.95,33,0.05,4655
spy_name,str,61258,99.99,8,0.01,60883
name_store,str,61263,100.00,3,0.00,60893
appid,int64,61266,100.00,0,0.00,61266
total_reviews,int64,61266,100.00,0,0.00,4092


...

,count,mean,std,min,25%,50%,75%,max,결측치 개수,왜도,첨도
positive,61266.0,838.673897,1.414126e+04,0.0,5.00,17.00,79.00,1373979.0,0,53.674034,3741.857666
negative,61266.0,103.772908,1.382147e+03,0.0,1.00,5.00,22.00,156649.0,0,60.598773,5045.758127
total_reviews,61266.0,942.446806,1.509816e+04,0.0,6.00,23.00,104.00,1409473.0,0,52.197823,3544.100505
price_spy,61266.0,656.000963,1.114242e+03,0.0,109.00,499.00,999.00,99998.0,0,22.403837,1279.497371
ccu,61266.0,25.027079,9.074564e+02,0.0,0.00,0.00,0.00,143870.0,0,102.798253,13481.229886
genre_count,61266.0,3.210508,1.248337e+00,1.0,2.00,3.00,4.00,16.0,0,0.945099,1.862996
positive_ratio_pct,60897.0,76.099132,2.357971e+01,0.0,65.26,82.05,94.61,100.0,369,-1.306704,1.523730
owners_lower_calc,61266.0,42538.928606,4.589211e+05,0.0,0.00,0.00,20000.00,50000000.0,0,59.629858,5240.682576
owners_upper_calc,61266.0,107766.624229,1.004997e+06,20000.0,20000.00,20000.00,50000.00,100000000.0,0,55.269531,4227.942318
owners_mid_calc,61266.0,75152.776418,7.309981e+05,10000.0,10000.00,10000.00,35000.00,75000000.0,0,56.454304,4510.278090



full_df의 날짜형 컬럼 요약


,컬럼명,결측치 개수,최소값,최대값,고유값 개수
0,release_date_dt,157,1997-06-30,2029-11-07,4598



full_df의 genres_list 리스트 항목 분포


,genres_list,개수,비율(%)
0,Indie,61218,99.92
1,Action,28149,45.95
2,Casual,27127,44.28
3,Adventure,26652,43.50
4,Simulation,12614,20.59
5,Strategy,12505,20.41
6,RPG,11621,18.97
7,Early Access,6340,10.35
8,Free To Play,3446,5.62
9,Sports,2514,4.10


## `reviews_df` 기술통계

In [48]:
check_basic_info(reviews_df, 'reviews_df')
check_id_duplicates(reviews_df, 'recommendationid', 'reviews_df')
check_category_summary(reviews_df, 'reviews_df', 'language')
check_category_summary(reviews_df, 'reviews_df', 'voted_up')
check_category_summary(reviews_df, 'reviews_df', 'steam_purchase')
check_category_summary(reviews_df, 'reviews_df', 'received_for_free')
check_category_summary(reviews_df, 'reviews_df', 'written_during_early_access')

check_numeric_summary(
    reviews_df,
    'reviews_df',
    cols=['votes_up', 'votes_funny', 'weighted_vote_score', 'comment_count',
          'author_num_games_owned', 'author_num_reviews',
          'author_playtime_forever', 'author_playtime_last_two_weeks',
          'author_playtime_at_review', 'author_playtime_forever_hour',
          'author_playtime_last_two_weeks_hour', 'author_playtime_at_review_hour',
          'review_char_len', 'review_word_count']
)

check_datetime_summary(
    reviews_df,
    'reviews_df',
    cols=['timestamp_created_dt', 'timestamp_updated_dt', 'author_last_played_dt']
)

check_text_summary(reviews_df, 'reviews_df', 'review')


reviews_df의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,13106
1,열 개수,33
2,중복 행 개수,6523


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
review,str,13104,99.98,2,0.02,6538
recommendationid,int64,13106,100.00,0,0.00,6553
timestamp_created,int64,13106,100.00,0,0.00,6553
timestamp_updated,int64,13106,100.00,0,0.00,6553
timestamp_created_dt,datetime64[s],13106,100.00,0,0.00,6553
timestamp_updated_dt,datetime64[s],13106,100.00,0,0.00,6553
author_last_played,int64,13106,100.00,0,0.00,6552
author_last_played_dt,datetime64[s],13106,100.00,0,0.00,6552
author_steamid,int64,13106,100.00,0,0.00,6479
weighted_vote_score,float64,13106,100.00,0,0.00,4023


[상위 5행]


...

,count,mean,std,min,25%,50%,75%,max,결측치 개수,왜도,첨도
votes_up,13106.0,39.240348,202.428208,0.000000,1.000000,4.000000,16.000000,5994.000000,0,14.767641,300.853385
votes_funny,13106.0,8.207844,75.367616,0.000000,0.000000,0.000000,1.000000,2719.000000,0,20.032327,513.793414
weighted_vote_score,13106.0,0.598915,0.122194,0.262922,0.500188,0.549109,0.650845,0.993874,0,1.338933,0.967254
comment_count,13106.0,0.709599,4.701845,0.000000,0.000000,0.000000,0.000000,130.000000,0,15.035367,288.213790
author_num_games_owned,13106.0,315.762552,1002.688897,0.000000,0.000000,54.000000,292.000000,29744.000000,0,13.070737,265.846012
author_num_reviews,13106.0,53.817641,295.971905,1.000000,5.000000,14.000000,40.000000,19938.000000,0,49.000611,3147.316052
author_playtime_forever,13106.0,2906.513200,12385.885850,5.000000,238.000000,652.000000,1822.000000,448786.000000,0,16.540270,407.352804
author_playtime_last_two_weeks,13106.0,11.300778,235.721234,0.000000,0.000000,0.000000,0.000000,14478.000000,0,44.092507,2381.589074
author_playtime_at_review,13106.0,1403.864795,7433.285469,5.000000,128.000000,326.000000,895.000000,263928.000000,0,20.010380,512.287082
author_playtime_forever_hour,13106.0,48.441846,206.431432,0.080000,3.970000,10.870000,30.370000,7479.770000,0,16.540275,407.353165



reviews_df의 날짜형 컬럼 요약


,컬럼명,결측치 개수,최소값,최대값,고유값 개수
0,timestamp_created_dt,0,2023-02-03 00:54:03,2026-01-08 17:26:21,6553
1,timestamp_updated_dt,0,2023-02-03 00:54:03,2026-04-19 05:17:37,6553
2,author_last_played_dt,0,2020-01-24 00:57:53,2026-04-21 05:10:35,6552



reviews_df의 review 텍스트 길이 요약


,항목,값
0,결측치 개수,2.00
1,평균 글자수,531.99
2,중앙값 글자수,244.00
3,최대 글자수,7999.00
4,최소 글자수,0.00


# 개별셀 확인